In [1]:
import tensorflow as tf

In [2]:
import keras

Using TensorFlow backend.


In [3]:
import numpy as np
import pandas as pd

from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
# import data set
df = pd.read_csv( 'bank.csv' )

In [5]:
df.shape

(10000, 14)

In [6]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
RowNumber          10000 non-null int64
CustomerId         10000 non-null int64
Surname            10000 non-null object
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [8]:
# check how many missing values
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [9]:
# drop ID column as it is useless for the model
if 'CustomerId' in df.columns:
    df.drop(['CustomerId','Surname'], axis=1, inplace = True)

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5000.500000,2886.895680,1.00,2500.75,5000.500,7500.2500,10000.00
CreditScore,10000.0,650.528800,96.653299,350.00,584.00,652.000,718.0000,850.00
Age,10000.0,38.921800,10.487806,18.00,32.00,37.000,44.0000,92.00
Tenure,10000.0,5.012800,2.892174,0.00,3.00,5.000,7.0000,10.00
Balance,10000.0,76485.889288,62397.405202,0.00,0.00,97198.540,127644.2400,250898.09
NumOfProducts,10000.0,1.530200,0.581654,1.00,1.00,1.000,2.0000,4.00
HasCrCard,10000.0,0.705500,0.455840,0.00,0.00,1.000,1.0000,1.00
IsActiveMember,10000.0,0.515100,0.499797,0.00,0.00,1.000,1.0000,1.00
EstimatedSalary,10000.0,100090.239881,57510.492818,11.58,51002.11,100193.915,149388.2475,199992.48
Exited,10000.0,0.203700,0.402769,0.00,0.00,0.000,0.0000,1.00


In [11]:
# Target Column Distribution
df.groupby(["Exited"]).count().T

Exited,0,1
RowNumber,7963,2037
CreditScore,7963,2037
Geography,7963,2037
Gender,7963,2037
Age,7963,2037
Tenure,7963,2037
Balance,7963,2037
NumOfProducts,7963,2037
HasCrCard,7963,2037
IsActiveMember,7963,2037


In [12]:
### 3.  Distinguish the feature and target set 

In [261]:
# Prepare X and Y
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [262]:
# convert categorical data into numeric data - create separate columns for each category - through get_dummies
X_df = pd.get_dummies(X, drop_first = True)

In [263]:
X_df.shape

(10000, 12)

In [264]:
X_df.head(5)

,RowNumber,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,1,619,42,2,0.00,1,1,1,101348.88,0,0,0
1,2,608,41,1,83807.86,1,0,1,112542.58,0,1,0
2,3,502,42,8,159660.80,3,1,0,113931.57,0,0,0
3,4,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,5,850,43,2,125510.82,1,1,1,79084.10,0,1,0


In [265]:
# encode class values (Y) as integers for binary classification
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(y)
encoded_y = encoder.transform(y)

In [266]:
# scale the input features
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_df = sc.fit_transform(X_df)

In [267]:
### Divide the data set into Train and test sets

In [268]:
from sklearn.model_selection import train_test_split

# Split the data into training and test set in the ratio of 70:30 respectively 
X_train, X_test, y_train, y_test = train_test_split(X_df, encoded_y, test_size = 0.3, random_state = 7)

In [269]:
### Normalize the train and test data 

In [270]:
from sklearn.preprocessing import Normalizer

transformer = Normalizer().fit(X_train)
X_train = transformer.transform(X_train)
X_test = transformer.transform(X_test)

In [272]:
### Initialize & build the model 

In [273]:
from keras.models import Sequential
from keras.layers import Dense

In [277]:
# create model
model_init = tf.keras.models.Sequential()

model_init.add(tf.keras.layers.Dense(12,input_dim=12,activation='relu'))
model_init.add(tf.keras.layers.Dense(8, activation='relu'))
model_init.add(tf.keras.layers.Dense(1, activation='sigmoid'))

# Compile model
model_init.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [278]:
# Fit the model
model_init.fit(X_train, y_train, epochs=10, batch_size=10)

Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 94us/sample - loss: 0.4846 - accuracy: 0.7933
Epoch 2/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.4300 - accuracy: 0.8063
Epoch 3/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.4197 - accuracy: 0.8173
Epoch 4/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.4096 - accuracy: 0.8247
Epoch 5/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3972 - accuracy: 0.8340
Epoch 6/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3827 - accuracy: 0.8424
Epoch 7/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3682 - accuracy: 0.8520
Epoch 8/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3570 - accuracy: 0.8567
Epoch 9/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3503 - accuracy: 0.8586
Epoch 10/10
7000/7000 [==========

In [279]:
# evaluate the model
score = model_init.evaluate(X_test, y_test, verbose=0)
# Print test accuracy
print('\n', 'Test accuracy for Initial Model :', score[1]*100)


 Test accuracy for Initial Model : 85.69999933242798


In [89]:
#### Optimize the model

In [280]:
# Use scikit-learn to grid search the batch size and epochs
import numpy
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
# Function to create model, required for KerasClassifier
def create_model(optimizer,activation):
    # create model
    model = tf.keras.models.Sequential()

    model.add(tf.keras.layers.Dense(12,activation=activation))
    model.add(tf.keras.layers.Dense(8, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    # Compile model
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# fix random seed for reproducibility
seed = 7
numpy.random.seed(seed)

# create model
model = KerasClassifier(build_fn=create_model, epochs=10, batch_size=10, verbose=0)
# define the grid search parameters
optimizers = ['sgd', 'adam', 'adamax']
#epochs = np.array([50, 100])
#batches = np.array([5, 10])
#param_grid = dict(optimizer=optimizers, nb_epoch=epochs, batch_size=batches)
activation = ['softmax', 'relu', 'tanh', 'sigmoid']
param_grid = dict(optimizer=optimizers,activation=activation)
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

C:\Users\Suchi\Anaconda3\lib\site-packages\sklearn\model_selection\_split.py:1978: FutureWarning: The default value of cv will change from 3 to 5 in version 0.22. Specify it explicitly to silence this warning.
  warnings.warn(CV_WARNING, FutureWarning)


Best: 0.853000 using {'activation': 'tanh', 'optimizer': 'adam'}
0.795429 (0.001435) with: {'activation': 'softmax', 'optimizer': 'sgd'}
0.813857 (0.007863) with: {'activation': 'softmax', 'optimizer': 'adam'}
0.798429 (0.005138) with: {'activation': 'softmax', 'optimizer': 'adamax'}
0.802857 (0.006695) with: {'activation': 'relu', 'optimizer': 'sgd'}
0.838143 (0.006493) with: {'activation': 'relu', 'optimizer': 'adam'}
0.817000 (0.003924) with: {'activation': 'relu', 'optimizer': 'adamax'}
0.809286 (0.003631) with: {'activation': 'tanh', 'optimizer': 'sgd'}
0.853000 (0.001233) with: {'activation': 'tanh', 'optimizer': 'adam'}
0.815857 (0.004564) with: {'activation': 'tanh', 'optimizer': 'adamax'}
0.795429 (0.001435) with: {'activation': 'sigmoid', 'optimizer': 'sgd'}
0.809714 (0.003691) with: {'activation': 'sigmoid', 'optimizer': 'adam'}
0.801571 (0.006360) with: {'activation': 'sigmoid', 'optimizer': 'adamax'}


In [287]:
bestmodel = grid.best_estimator_.model
# evaluate the model
score = bestmodel.evaluate(X_test, y_test, verbose=0)
# Print test accuracy
print('\n', 'Test accuracy for Optimized Model :', score[1]*100)


 Test accuracy for Optimized Model : 85.9333336353302


In [288]:
y_pred = bestmodel.predict(X_test)
print(y_pred)

[[0.50199276]
 [0.06657279]
 [0.02733326]
 ...
 [0.00971746]
 [0.03082478]
 [0.21089348]]


In [ ]:
### Predict the results using 0.5 as a threshold

In [289]:
# Predict Class 
class_pred = []
for val in y_pred:
    if val> 0.5: # use Threshold of 0.5
        class_pred.append(1)
    else:
        class_pred.append(0)
print(class_pred)

[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [290]:
from sklearn.metrics import confusion_matrix, accuracy_score

In [293]:
print('\n', 'Confusion Matrix of Optimized Model :')
confusion_matrix(y_test,class_pred)


 Confusion Matrix of Optimized Model :


array([[2292,  103],
       [ 319,  286]], dtype=int64)

In [292]:
# Print accuracy score
print('\n', 'Accuracy Score of Optimized Model :', accuracy_score(y_test,class_pred)*100)


 Accuracy Score of Optimized Model : 85.93333333333332
